# Step 01 — Storywrangler API Query (build raw run)

This notebook reads `config.WORD_FORMS_ALL`, fetches Storywrangler time series per query-language pair, and creates a timestamped raw run folder.

## Input
- `config.WORD_FORMS_ALL` (default: `data/input/all_forms_manuscript_version.csv`)

## Outputs (`data/raw/run_YYYYMMDD_HHMMSS/`)
- `jsons/{Language}_{query}.json` — successful responses
- `jsons/{Language}_{query}_empty.json` — no usable API data
- `timeseries_data.csv` — API data only (`date`, `query`, `language_ISO`, `count`, `count_no_rt`, `rank`, `freq`, `freq_no_rt`)
- `query_metadata.csv` — lookup table (`query`, `ISO`, `Language`, plus metadata columns)

## Notes
- Existing JSONs are reused from earlier runs when available (to reduce API requests).
- To force re-fetch for one query, delete its JSON in `data/raw/run_*/jsons/` first.
- If you switch input files, update `config.CURRENT_INPUT_NAME` in `config.py`, restart kernel, and rerun.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
from config import (
    WORD_FORMS_ALL, RAW_DIR,
    API_BASE_URL, API_DELAY_MIN, API_DELAY_MAX
)

import requests
import json
import pandas as pd
import time
import random
from urllib.parse import quote
from tqdm.notebook import tqdm

from datetime import datetime
import shutil
from pathlib import Path

# Create a timestamped run folder inside RAW_DIR. To reuse an existing run folder,
# set RUN_SUBDIR_NAME to the folder name you want (e.g. 'run_20230101_120000') before running.
RUN_SUBDIR_NAME = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = RAW_DIR / RUN_SUBDIR_NAME
JSONS_DIR = RUN_DIR / "jsons"  # JSONs stored in jsons/ subfolder
JSONS_DIR.mkdir(parents=True, exist_ok=True)

print('Word forms input  :', WORD_FORMS_ALL)
print('Run folder        :', RUN_DIR)
print('JSONs folder      :', JSONS_DIR)
print('CSV will be saved :', RUN_DIR / 'search_results_raw.csv')

## 1. Load word forms

In [ ]:
# Load the full word list -- edit word_forms_all.csv to change queried words
df_words = pd.read_csv(WORD_FORMS_ALL)

print(f'Total word forms to query: {len(df_words)}')
print(f'Languages covered: {df_words["Language"].nunique()}')
print()
df_words.head(5)

## 2. Run API queries

Each row in `word_forms_all.csv` leads to one API call and one JSON saved to `data/raw/json_responses/`.

**Skip logic:** if the JSON file already exists, the row is skipped.
Delete a file (or the whole folder) to force a re-fetch.

In [ ]:
def fetch_word(word: str, lang_iso: str) -> dict:
    """
    Call Storywrangler API for one word+language (uses ISO for the API).
    Returns parsed JSON dict, or None on error.
    """
    encoded_word = quote(word, safe='')  # encodes # → %23, spaces, etc.
    url = f"{API_BASE_URL}/{encoded_word}?language={lang_iso}&gapped=true&gapped=false"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return response.json()
    except (requests.RequestException, ValueError, json.JSONDecodeError) as e:
        tqdm.write(f'  ERROR [{lang_iso}] "{word}": {e}')
        return None


def clean_lang_name(lang: str) -> str:
    """Make a filesystem-safe language prefix from a full language name."""
    if not isinstance(lang, str) or not lang:
        return ''
    return (lang.replace(' ', '_').replace('/', '_').replace('\\', '_').replace('#', 'HASHTAG_'))

def safe_filename(lang_prefix: str, word: str) -> str:
    """Create canonical filename using cleaned language name (fallback to ISO if empty)."""
    clean_word = word.replace('/', '_').replace('\\', '_').replace('#', 'HASHTAG_')
    return f"{lang_prefix}_{clean_word}.json"


def is_empty_response(r: dict, word: str) -> bool:
    """True if the API returned no usable data for this word."""
    return not r.get('data') or not r['data'].get(word)


def find_existing_anywhere(lang_iso: str, lang_name: str, word: str):
    """Search RAW_DIR recursively for matching files.
    Tries: cleaned language name prefix, ISO prefix, literal/# and percent-encoded variants for the word, and a loose-stem fallback.
    Returns Path or None.
    """
    # prepare prefixes
    lang_prefix = clean_lang_name(lang_name) if lang_name else ''
    variants_prefixes = []
    if lang_prefix:
        variants_prefixes.append(lang_prefix)
    if lang_iso:
        variants_prefixes.append(lang_iso)

    # possible word variants
    cleaned_word = word.replace('/', '_').replace('\\', '_').replace('#', 'HASHTAG_')
    encoded = quote(word, safe='')
    literal_hash = word
    word_variants = [cleaned_word, cleaned_word.replace('HASHTAG_', '#'), encoded, literal_hash]

    # try exact filename variants
    for pfix in variants_prefixes:
        for wv in word_variants:
            candidate = f"{pfix}_{wv}.json"
            p = next(RAW_DIR.rglob(candidate), None)
            if p:
                return p

            # _empty variant
            candidate_e = f"{pfix}_{wv}_empty.json"
            p2 = next(RAW_DIR.rglob(candidate_e), None)
            if p2:
                return p2

    # loose fallback: scan files that contain either prefix or word tokens
    tokens = set([lang_iso or '', lang_prefix])
    for p in RAW_DIR.rglob('*.json'):
        # only consider files inside a json_responses folder (project layout)
        if 'json_responses' not in p.parts:
            continue
        stem = p.stem
        if any(t and t in stem for t in tokens) or any(wv in stem for wv in word_variants):
            return p

    return None

In [ ]:
dataframes = []
skipped = 0
fetched  = 0
errors   = 0

# build ISO -> language mapping from the master input so we can map old ISO prefixes to full names
iso_to_lang = dict(df_words[['ISO', 'Language']].drop_duplicates().values)

for _, row in tqdm(df_words.iterrows(), total=len(df_words), desc='Querying API'):
    word      = str(row['Query'])
    lang_iso  = str(row['ISO']) if 'ISO' in row and not pd.isna(row['ISO']) else ''
    lang_name = str(row['Language']) if 'Language' in row and not pd.isna(row['Language']) else ''
    lang_prefix = clean_lang_name(lang_name) or lang_iso

    # Filenames use full language-name prefix
    fname       = safe_filename(lang_prefix, word)
    fpath       = JSONS_DIR / fname
    fpath_empty = JSONS_DIR / fname.replace('.json', '_empty.json')

    # If file already saved in THIS run folder, skip.
    if fpath.exists() or fpath_empty.exists():
        skipped += 1
        continue

    # If file exists in any older run folder, copy it into this run's jsons/ folder (rename to canonical name) and skip fetching.
    existing = find_existing_anywhere(lang_iso, lang_name, word)
    if existing is not None:
        try:
            # copy and rename to canonical filename using the language-name prefix
            target = JSONS_DIR / fname
            shutil.copy2(existing, target)
            tqdm.write(f'  [FOUND] Copied existing {existing.name} → {target.name}')
            # if copied file has data for this word, load it into this session's dataframes
            try:
                with open(target, 'r', encoding='utf-8') as tf:
                    jr = json.load(tf)
                if jr.get('data') and jr['data'].get(word):
                    df = pd.DataFrame(jr['data'][word])
                    df['language'] = lang_name
                    df['language_ISO'] = lang_iso
                    df['query'] = word
                    dataframes.append(df)
            except Exception:
                pass
        except Exception as e:
            tqdm.write(f'  [COPY ERROR] could not copy {existing}: {e}')
        skipped += 1
        continue

    # Fetch from API (use ISO for the API call)
    result = fetch_word(word, lang_iso)
    if result is None:
        errors += 1
        continue

    # Save into jsons/ folder (using language-name prefix)
    empty     = is_empty_response(result, word)
    save_path = fpath_empty if empty else fpath

    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    if empty:
        tqdm.write(f'  [EMPTY] [{lang_prefix}] "{word}" -- saved as _empty.json')
    else:
        df = pd.DataFrame(result['data'][word])
        df['language']     = lang_name
        df['language_ISO'] = lang_iso
        df['query']        = word
        dataframes.append(df)
        fetched += 1

    # Polite delay
    time.sleep(random.uniform(API_DELAY_MIN, API_DELAY_MAX))

print(f'\nDone.  Fetched: {fetched}  |  Skipped (already saved or copied): {skipped}  |  Errors: {errors}')

## 3. Combine and save raw results

In [ ]:
# Load JSONs from THIS run folder only so the CSV reflects only this run
print('Loading JSONs from current run folder...')
all_frames = []

# Build ISO -> Language mapping from the master input so we can resolve ISO prefixes to full names
iso_to_lang = dict(df_words[['ISO', 'Language']].drop_duplicates().values)
lang_to_iso = {v: k for k, v in iso_to_lang.items()}  # reverse mapping for later

# Only read JSONs from this run's jsons/ folder
for json_file in sorted(JSONS_DIR.rglob('*.json')):
    if '_empty' in json_file.stem:
        continue

    with open(json_file, 'r', encoding='utf-8') as f:
        r = json.load(f)

    if not r.get('data'):
        continue

    # filename prefix may be full language name or an ISO; try to resolve to full language name
    prefix = json_file.stem.split('_', 1)[0]
    language = iso_to_lang.get(prefix, prefix.replace('_', ' '))
    language_iso = lang_to_iso.get(language, prefix)

    for word, word_data in r['data'].items():
        df = pd.DataFrame(word_data)
        df['query'] = word
        df['language_ISO'] = language_iso
        all_frames.append(df)

if all_frames:
    # ===== TIMESERIES TABLE (API data only - normalized) =====
    timeseries = pd.concat(all_frames, ignore_index=True)
    timeseries['date'] = pd.to_datetime(timeseries['date'])

    # Keep only API response columns + query keys
    ts_cols = ['date', 'query', 'language_ISO', 'count', 'count_no_rt', 'rank', 'freq', 'freq_no_rt']
    timeseries = timeseries[[c for c in ts_cols if c in timeseries.columns]]

    # Save timeseries (compact, no redundant metadata)
    ts_csv = RUN_DIR / 'timeseries_data.csv'
    timeseries.to_csv(ts_csv, index=False)

    print(f'✓ Saved: {ts_csv}')
    print(f'  Rows: {len(timeseries):,}  |  Unique queries: {timeseries["query"].nunique()}  |  Languages: {timeseries["language_ISO"].nunique()}')

    # ===== METADATA TABLE (lookup table: ISO = abbreviation, Language = full name) =====
    # All columns except Query, Language, ISO are treated as metadata
    metadata_cols = [c for c in df_words.columns if c not in ['Query', 'Language', 'ISO']]

    query_metadata = df_words[['Query', 'Language', 'ISO'] + metadata_cols].copy()
    # Rename: Query -> query  (ISO and Language stay as-is)
    query_metadata = query_metadata.rename(columns={'Query': 'query'})

    # Reorder columns: query, ISO, Language, then remaining metadata
    cols_ordered = ['query', 'ISO', 'Language'] + metadata_cols
    query_metadata = query_metadata[[c for c in cols_ordered if c in query_metadata.columns]]

    # Remove duplicates (in case input CSV has duplicates)
    query_metadata = query_metadata.drop_duplicates(subset=['query', 'ISO'])

    # Save metadata lookup table
    meta_csv = RUN_DIR / 'query_metadata.csv'
    query_metadata.to_csv(meta_csv, index=False)

    print(f'✓ Saved: {meta_csv}')
    print(f'  Rows: {len(query_metadata):,}  |  Unique query+language pairs')
    print(f'  Columns: {list(query_metadata.columns)[:6]}...')

    # ===== SUMMARY =====
    print(f'\n📊 Storage savings:')
    ts_size_mb = ts_csv.stat().st_size / (1024**2)
    meta_size_mb = meta_csv.stat().st_size / (1024**2)
    total_size = ts_size_mb + meta_size_mb
    print(f'  Timeseries: {ts_size_mb:.1f} MB')
    print(f'  Metadata:   {meta_size_mb:.1f} MB')
    print(f'  Total:      {total_size:.1f} MB (normalized)')
    print(f'\n✓ Join key: timeseries[language_ISO] == metadata[ISO]')
    print()
    display(timeseries.head(3))
else:
    print('No data found -- check that the API calls above ran successfully.')

## 4. Sanity check -- which words returned no data?

Words that returned `_empty.json` are listed here.
These may need alternative spellings in `word_forms_all.csv`.

In [ ]:
empty_files = sorted(JSONS_DIR.rglob('*_empty.json'))

if empty_files:
    print(f'{len(empty_files)} word(s) returned no data from the API:\n')
    # Build language ISO → name mapping for output
    iso_to_lang_map = dict(df_words[['ISO', 'Language']].drop_duplicates().values)
    
    for f in empty_files:
        stem  = f.stem.replace('_empty', '')
        parts = stem.split('_', 1)
        lang_key  = parts[0]
        word  = parts[1] if len(parts) > 1 else '?'
        # Resolve prefix to full language name
        lang_name = iso_to_lang_map.get(lang_key, lang_key.replace('_', ' '))
        print(f'  [{lang_name}]  {word}')
else:
    print('All words returned data -- no empty responses.')